# Test LLM Manager Function

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Import the necessary modules
from src.services.llm_manager import LLMManager, llm_manager
from src.config import get_db, SessionLocal
# Import essential models (the relationship issue should be fixed by importing ChatUser in __init__.py)
from src.models.tenant import Tenant
from src.models.llm_model import LLMModel
from src.models.tenant_llm_config import TenantLLMConfig
from uuid import UUID
import redis
from sqlalchemy.orm import Session
from src.utils.encryption import decrypt_api_key

# Initialize LLM Manager
llm_manager = LLMManager()

c:\Users\lucy.le\Downloads\chatbot_lastest\itl_chatbot_works\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Function to list all available tenants
def list_tenants():
    """List all tenants in the database with their IDs"""
    db_gen = get_db()
    db: Session = next(db_gen)
    try:
        tenants = db.query(Tenant).all()
        print("Available tenants:")
        for tenant in tenants:
            print(f"  - Name: {tenant.name}, Domain: {tenant.domain}, ID: {tenant.tenant_id}")
        return tenants
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        raise e
    finally:
        next(db_gen, None)  # This closes the session


In [3]:
# Function to list all available LLM models for a tenant
def list_tenant_llm_configs():
    """List all tenant LLM configurations with their details"""
    db_gen = get_db()
    db: Session = next(db_gen)
    try:
        configs = db.query(TenantLLMConfig).all()
        print("Tenant LLM configurations:")
        for config in configs:
            tenant = db.query(Tenant).filter(Tenant.tenant_id == config.tenant_id).first()
            llm_model = db.query(LLMModel).filter(LLMModel.llm_model_id == config.llm_model_id).first()
            print(f"  - Tenant: {tenant.name if tenant else 'N/A'}, Model: {llm_model.model_name if llm_model else 'N/A'}, Provider: {llm_model.provider if llm_model else 'N/A'}, Rate Limits: {config.rate_limit_rpm} RPM, {config.rate_limit_tpm} TPM")
        return configs
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        raise e
    finally:
        next(db_gen, None)  # This closes the session


In [4]:
# Test get_llm_for_tenant function
def test_get_llm_for_tenant(tenant_id: str, llm_model_id: str = None):
    """
    Test function for get_llm_for_tenant
    
    Args:
        tenant_id: The tenant UUID to test with
        llm_model_id: Optional specific model ID, otherwise uses tenant's default
        
    Returns:
        dict: Contains agent, model, and decrypted API key information
    """
    try:
        # Get database session
        db_gen = get_db()
        db: Session = next(db_gen)
        
        print(f"Testing get_llm_for_tenant with tenant_id: {tenant_id}, llm_model_id: {llm_model_id}")
        
        # Get LLM client for tenant
        llm_client = llm_manager.get_llm_for_tenant(db, tenant_id, llm_model_id)
        
        # Get tenant config to retrieve API key and other details
        tenant_config = (
            db.query(TenantLLMConfig).filter(TenantLLMConfig.tenant_id == tenant_id).first()
        )
        
        if not tenant_config:
            raise ValueError(f"No LLM configuration found for tenant {tenant_id}")
        
        # Use specified model or tenant's default
        model_id = llm_model_id or tenant_config.llm_model_id
        
        # Get LLM model details
        llm_model = db.query(LLMModel).filter(LLMModel.llm_model_id == model_id).first()
        
        if not llm_model:
            raise ValueError(f"LLM model {model_id} not found")
        
        # Decrypt API key
        decrypted_api_key = decrypt_api_key(tenant_config.encrypted_api_key)
        
        # Return the details
        result = {
            "agent": llm_client,
            "model": {
                "model_name": llm_model.model_name,
                "provider": llm_model.provider,
                "is_active": llm_model.is_active
            },
            "api_key": decrypted_api_key,  # Note: This is the decrypted API key
            "llm_client_type": type(llm_client).__name__
        }
        
        print(f"Successfully retrieved LLM client:")
        print(f"  - Type: {result['llm_client_type']}")
        print(f"  - Model: {result['model']['model_name']}")
        print(f"  - Provider: {result['model']['provider']}")
        print(f"  - Active: {result['model']['is_active']}")
        
        # Close database session
        next(db_gen, None)  # This closes the session
        
        return result
        
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        # Close database session in case of error
        try:
            next(db_gen, None)
        except:
            pass
        raise e


In [5]:
# First, list all available tenants to get the tenant IDs to test with
tenants = list_tenants()
print()
list_tenant_llm_configs()

2025-11-19 14:55:49,437 INFO sqlalchemy.engine.Engine select pg_catalog.version()


select pg_catalog.version()


2025-11-19 14:55:49,441 INFO sqlalchemy.engine.Engine [raw sql] {}


[raw sql] {}


2025-11-19 14:55:49,454 INFO sqlalchemy.engine.Engine select current_schema()


...

2025-11-19 14:55:49,695 INFO sqlalchemy.engine.Engine [cached since 0.1052s ago] {'llm_model_id_1': UUID('b2c3d4e5-f6a7-4859-a5a7-b2c3d4e5f6a7'), 'param_1': 1}


[cached since 0.1052s ago] {'llm_model_id_1': UUID('b2c3d4e5-f6a7-4859-a5a7-b2c3d4e5f6a7'), 'param_1': 1}


  - Tenant: Vela, Model: openai/gpt-4o-mini, Provider: openrouter, Rate Limits: 60 RPM, 10000 TPM
2025-11-19 14:55:49,704 INFO sqlalchemy.engine.Engine ROLLBACK


ROLLBACK


[<TenantLLMConfig(tenant_id=1193a40f-1d03-4ecd-a601-901a55589f56, llm_model_id=a1b2c3d4-e5f6-4748-9394-a1b2c3d4e5f6)>,
 <TenantLLMConfig(tenant_id=3105b788-b5ff-4d56-88a9-532af4ab4ded, llm_model_id=c5565f19-2d19-4384-82c2-505a7ceac625)>,
 <TenantLLMConfig(tenant_id=d19a8569-a01f-4026-91b2-9da41f2e0cc2, llm_model_id=15f4efa9-46f8-40d2-9cfd-e871977a5634)>,
 <TenantLLMConfig(tenant_id=7319e693-a4c9-4023-be86-e50184e80abf, llm_model_id=b2c3d4e5-f6a7-4859-a5a7-b2c3d4e5f6a7)>]

In [ ]:
# Example usage of the test function
# Use the tenant ID from the list above
# tenant_id = "replace-with-actual-tenant-id-from-above-list"

# Example test (uncomment and replace with actual tenant ID):
# result = test_get_llm_for_tenant(tenant_id)
# print(result)

In [ ]:
# Test with default model (no specific llm_model_id)
# result = test_get_llm_for_tenant(tenant_id, None)

In [6]:
# Test with specific model
result = test_get_llm_for_tenant('3105b788-b5ff-4d56-88a9-532af4ab4ded', "specific-model-id")

Testing get_llm_for_tenant with tenant_id: 3105b788-b5ff-4d56-88a9-532af4ab4ded, llm_model_id: specific-model-id
2025-11-19 14:58:37,757 INFO sqlalchemy.engine.Engine BEGIN (implicit)


BEGIN (implicit)


2025-11-19 14:58:37,762 INFO sqlalchemy.engine.Engine SELECT tenant_llm_configs.config_id AS tenant_llm_configs_config_id, tenant_llm_configs.tenant_id AS tenant_llm_configs_tenant_id, tenant_llm_configs.llm_model_id AS tenant_llm_configs_llm_model_id, tenant_llm_configs.encrypted_api_key AS tenant_llm_configs_encrypted_api_key, tenant_llm_configs.rate_limit_rpm AS tenant_llm_configs_rate_limit_rpm, tenant_llm_configs.rate_limit_tpm AS tenant_llm_configs_rate_limit_tpm, tenant_llm_configs.created_at AS tenant_llm_configs_created_at, tenant_llm_configs.updated_at AS tenant_llm_configs_updated_at 
FROM tenant_llm_configs 
WHERE tenant_llm_configs.tenant_id = %(tenant_id_1)s::UUID 
 LIMIT %(param_1)s


SELECT tenant_llm_configs.config_id AS tenant_llm_configs_config_id, tenant_llm_configs.tenant_id AS tenant_llm_configs_tenant_id, tenant_llm_configs.llm_model_id AS tenant_llm_configs_llm_model_id, tenant_llm_configs.encrypted_api_key AS tenant_llm_configs_encrypted_api_key, tenant_llm_configs.rate_limit_rpm AS tenant_llm_configs_rate_limit_rpm, tenant_llm_configs.rate_limit_tpm AS tenant_llm_configs_rate_limit_tpm, tenant_llm_configs.created_at AS tenant_llm_configs_created_at, tenant_llm_configs.updated_at AS tenant_llm_configs_updated_at 
FROM tenant_llm_configs 
WHERE tenant_llm_configs.tenant_id = %(tenant_id_1)s::UUID 
 LIMIT %(param_1)s


2025-11-19 14:58:37,765 INFO sqlalchemy.engine.Engine [generated in 0.00311s] {'tenant_id_1': '3105b788-b5ff-4d56-88a9-532af4ab4ded', 'param_1': 1}


...

2025-11-19 14:58:37,782 INFO sqlalchemy.engine.Engine [cached since 168.2s ago] {'llm_model_id_1': 'specific-model-id', 'param_1': 1}


[cached since 168.2s ago] {'llm_model_id_1': 'specific-model-id', 'param_1': 1}


Error occurred: (psycopg2.errors.InvalidTextRepresentation) invalid input syntax for type uuid: "specific-model-id"
LINE 3: WHERE llm_models.llm_model_id = 'specific-model-id'::UUID 
                                        ^

[SQL: SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models 
WHERE llm_models.llm_model_id = %(llm_model_id_1)s::UUID 
 LIMIT %(param_1)s]
[parameters: {'llm_model_id_1': 'specific-model-id', 'param_1': 1}]
(Background on this error at: https://sqlalche.me/e/20/9h9h)
2025-11-19 14:58:37,793 INFO sqlalche

ROLLBACK


DataError: (psycopg2.errors.InvalidTextRepresentation) invalid input syntax for type uuid: "specific-model-id"
LINE 3: WHERE llm_models.llm_model_id = 'specific-model-id'::UUID 
                                        ^

[SQL: SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models 
WHERE llm_models.llm_model_id = %(llm_model_id_1)s::UUID 
 LIMIT %(param_1)s]
[parameters: {'llm_model_id_1': 'specific-model-id', 'param_1': 1}]
(Background on this error at: https://sqlalche.me/e/20/9h9h)

In [ ]:
from cryptography.fernet import Fernet
from dotenv import load_dotenv
import os

# Load .env from src folder
load_dotenv("src/.env")

FERNET_KEY = os.getenv("FERNET_KEY")

def encrypt_api_key(api_key: str) -> str:
    """Encrypt API key for database storage."""
    cipher = Fernet(FERNET_KEY.encode())
    return cipher.encrypt(api_key.encode()).decode()

# Usage
api_key = "your-api-key-here"
encrypted = encrypt_api_key(api_key)
print(f"Encrypted:\n{encrypted}")